In [1]:
import pandas as pd
import dotenv
from datetime import datetime, timedelta, timezone

dotenv.load_dotenv()

True

In [2]:
with open('data/mentalhealth_submissions.jsonl', 'r', encoding='utf-8') as f:
    for i in range(5):
        print(f.readline())

{"archived":true,"author":"robertduddeley","author_flair_background_color":null,"author_flair_css_class":null,"author_flair_richtext":[],"author_flair_text":null,"author_flair_text_color":null,"author_flair_type":"text","brand_safe":false,"can_gild":true,"contest_mode":false,"created_utc":1213296391,"distinguished":null,"domain":"addadhdblog.com","edited":false,"gilded":0,"hidden":false,"hide_score":false,"id":"6n4q9","is_crosspostable":true,"is_reddit_media_domain":false,"is_self":false,"is_video":false,"link_flair_css_class":null,"link_flair_richtext":[],"link_flair_text":null,"link_flair_text_color":"dark","link_flair_type":"text","locked":false,"media":null,"media_embed":{},"no_follow":true,"num_comments":0,"num_crossposts":0,"over_18":false,"parent_whitelist_status":null,"permalink":"\/r\/mentalhealth\/comments\/6n4q9\/women_with_add_are_more_impaired\/","retrieved_on":1522694976,"rte_mode":"markdown","score":1,"secure_media":null,"secure_media_embed":{},"selftext":"","send_replie

In [3]:
file_path = "data/mentalhealth_submissions.jsonl"
chunksize = 100_000  # tune for your RAM

# compute cutoff for "last 2 years" from now
cutoff_dt = datetime.now(timezone.utc) - timedelta(days=2*365)
cutoff_ts = cutoff_dt.timestamp()

frames = []

for chunk in pd.read_json(file_path, lines=True, chunksize=chunksize):
    # keep only last 2 years
    chunk = chunk[chunk["created_utc"] >= cutoff_ts]

    # drop deleted/removed posts
    chunk = chunk[~chunk["selftext"].isin(["[deleted]", "[removed]"])]
    chunk = chunk[~chunk["author"].isin(["[deleted]", "[removed]"])]

    if len(chunk):
        frames.append(chunk)

time_filtered_df = pd.concat(frames, ignore_index=True)
clean_df = time_filtered_df[
    [
        "author",
        "selftext",
        "created_utc",
        "title",
        "permalink",
        "id",
        "subreddit",
        "subreddit_id",
        "link_flair_text",
        "retrieved_on",
    ]
].dropna()

# add t3_ prefix to id and rename to post_id
# chromadb uses "id" to identify documents
clean_df["post_id"] = "t3_" + clean_df["id"].astype(str)
clean_df = clean_df.drop(columns=["id"])

print(len(clean_df), "rows in last 2 years after filtering")

207438 rows in last 2 years after filtering


## Inspecting Data

Huh. Turns out there's a lot of entries with no text. Let's get rid of them. Only keep entries with 100+ words.

In [4]:
selftext_lengths = clean_df["selftext"].fillna("").apply(lambda x: len(str(x).split()))
title_lengths = clean_df["title"].fillna("").apply(lambda x: len(str(x).split()))

zero_len_df = clean_df[selftext_lengths == 0]

print(f"Number of posts with 0-word selftext: {len(zero_len_df)}")
print(zero_len_df[["author", "title", "selftext", "permalink"]].head(5).to_string(index=False))

Number of posts with 0-word selftext: 1008
              author                                                                                                          title selftext                                                                            permalink
    HungryAnswer1776 It feels like I'm living a lie fabricated by the people around me and I don't know how to know that's not true             /r/mentalhealth/comments/1cmexbx/it_feels_like_im_living_a_lie_fabricated_by_the/
  Beautiful-Money343                                                     I AM SICK OF RE-STARTING MY WEIGHTLOSS JOURNEY !!!!! DAY-3          /r/mentalhealth/comments/1ev3bu9/i_am_sick_of_restarting_my_weightloss_journey_day3/
     Glass_Sort_5661                  Your mental health is a priority. Your happiness is essential. Your self-care is a necessity.             /r/mentalhealth/comments/1evx05r/your_mental_health_is_a_priority_your_happiness/
Subject-Channel-8959                                 

In [5]:
clean_df = clean_df[selftext_lengths > 100]
print(f"{len(clean_df)} rows remain after dropping posts with 100-word selftext")

143282 rows remain after dropping posts with 100-word selftext


 turns chromadb limits document sizes to 16kb and we have text documents in here that's larger than 16kb.

we wil find out whats the max length of a title (since our document combines title with body text), and then set a cap to how long the body can be, then truncate

In [6]:
max_title_len = clean_df["title"].fillna("").apply(lambda x: len(str(x))).max()
print(f"Maximum title length in characters: {max_title_len}")

Maximum title length in characters: 300


In [7]:
# ChromaDB limits documents to 16KB (~16k chars). Truncate selftext so title + " | " + selftext <= 16000
MAX_DOC_CHARS = 16000
SEPARATOR = " | "

before_len = clean_df["title"].fillna("").str.len() + len(SEPARATOR) + clean_df["selftext"].fillna("").str.len()
n_truncated = (before_len > MAX_DOC_CHARS).sum()

def truncate_selftext(row):
    title = str(row["title"]) if pd.notna(row["title"]) else ""
    selftext = str(row["selftext"]) if pd.notna(row["selftext"]) else ""
    max_selftext_len = MAX_DOC_CHARS - len(title) - len(SEPARATOR)
    if len(selftext) > max_selftext_len:
        return selftext[:max_selftext_len]
    return selftext

clean_df["selftext"] = clean_df.apply(truncate_selftext, axis=1)
print(f"Documents truncated to {MAX_DOC_CHARS} chars. Rows affected: {n_truncated}")

# Show an example of a truncated document
if n_truncated > 0:
    example_idx = before_len[before_len > MAX_DOC_CHARS].index[0]
    row = clean_df.loc[example_idx]
    doc = row["title"] + SEPARATOR + row["selftext"]
    print(f"\nExample truncated doc (post_id={row['post_id']}, orig length={int(before_len.loc[example_idx])} chars):")
    print(f"  Total length now: {len(doc)} chars")
    print(f"  Title: {row['title'][:80]}...")
    print(f"  Selftext (last 400 chars, where cut): ...{row['selftext'][-400:]}")

Documents truncated to 16000 chars. Rows affected: 13

Example truncated doc (post_id=t3_1bciedd, orig length=18886 chars):
  Total length now: 16000 chars
  Title: I cannot figure out how to let my ex partner go and break an NPD cycle...
  Selftext (last 400 chars, where cut): ...ften short tempered and quick to snap back so I know that I wasn't the best to be around I had just felt like our relationship was stalling out again and I couldnt continue to try and look past the unwillingness to talk and resolve issues so i just avoided them by avoiding her when I got home. February 8th I brought up to her how we should move back into one apartment since we were paying for 2 at


## Cost and Size Checks

In [92]:
import tiktoken

encoding = tiktoken.encoding_for_model("text-embedding-3-small")

def count_selftext_tokens(df, encoding):
    total_tokens = 0
    titles = df["title"].fillna("")
    selftexts = df["selftext"].fillna("")

    for i, (title, selftext) in enumerate(zip(titles, selftexts), 1):
        text = f"{title} | {selftext}"
        num_tokens = len(encoding.encode(str(text)))
        total_tokens += num_tokens
        if i % 20000 == 0:
            print(f"Token sum after {i} rows: {total_tokens:,}")

    print(f"Total tokens in df title|selftext: {total_tokens:,}")
    # openai text-embedding-3-small cost is $0.02 per million tokens
    cost = total_tokens / 1000000 * 0.02
    print(f"Cost: ${cost:.2f}")
    return

count_selftext_tokens(clean_df, encoding)


Token sum after 20000 rows: 7,327,342
Token sum after 40000 rows: 13,610,777
Token sum after 60000 rows: 20,013,812
Token sum after 80000 rows: 26,540,445
Token sum after 100000 rows: 33,255,373
Token sum after 120000 rows: 40,014,174
Token sum after 140000 rows: 46,754,793
Total tokens in df title|selftext: 47,997,736
Cost: $0.96


In [93]:
BYTES_PER_VALUE = 4   # float32


def estimate_vector_store_size(num_vectors, dim, bytes_per_value=BYTES_PER_VALUE):
    total_bytes = num_vectors * dim * bytes_per_value
    total_mb = total_bytes / (1024 ** 2)
    total_gb = total_bytes / (1024 ** 3)
    print(f"Estimated raw embedding storage for {num_vectors:,} vectors:")
    print(f"~{total_mb:,.2f} MB (~{total_gb:,.2f} GB) of float32 embeddings")
    return


num_vectors = len(clean_df)
estimate_vector_store_size(num_vectors, 1024)
estimate_vector_store_size(num_vectors, 512)
estimate_vector_store_size(num_vectors, 256)

Estimated raw embedding storage for 143,680 vectors:
~561.25 MB (~0.55 GB) of float32 embeddings
Estimated raw embedding storage for 143,680 vectors:
~280.62 MB (~0.27 GB) of float32 embeddings
Estimated raw embedding storage for 143,680 vectors:
~140.31 MB (~0.14 GB) of float32 embeddings


# Create ChromaDB

In [8]:
import chromadb
import os

client = chromadb.CloudClient(
  api_key=os.environ.get("CHROMA_API_KEY"),
)

In [9]:
from chromadb.utils.embedding_functions import OpenAIEmbeddingFunction

COLLECTION_NAME = "reddit-bot-vdb-2"

collection = client.create_collection(
    name=COLLECTION_NAME,
    embedding_function=OpenAIEmbeddingFunction(
        model_name="text-embedding-3-small",
        api_key=os.environ.get("OPENAI_API_KEY"),
    ),
)

collection = client.get_collection(COLLECTION_NAME)

## Small batch test upload to chroma

In [10]:
chroma_test_df = clean_df.head(3)

In [11]:

documents = (
    chroma_test_df["title"]
    + " | "
    + chroma_test_df["selftext"]
).tolist()

ids = chroma_test_df["post_id"].astype(str).tolist()

metadatas = chroma_test_df[
    [
        "author",
        "created_utc",
        "permalink",
        "subreddit",
        "subreddit_id",
        "link_flair_text",
        "retrieved_on",
        "post_id",   
    ]
].to_dict(orient="records")

print(documents[:1])
print(ids[:3])
print(metadatas[:2])

["Venting about my shitty life | I just feel like such a fucking loser, I have 1 actual friend in this dumb fucking town I don't even go to a normal school and the 1 friend I have I feel like she always makes up excuses not to hangout or have sleepovers what the actual fuck is wrong with me and what did I do wrong I'm just so fucked up in so many ways I hate it and I hate myself so fucking much I just Wana take a fuck ton of pills and kill myself and get it done and over with. Not even my mother likes me I'm so fuckimg sick of everyone and everything "]
['t3_1b51qrm', 't3_1b520qi', 't3_1b5245s']
[{'author': 'd3ath-n0te', 'created_utc': 1709421179, 'permalink': '/r/mentalhealth/comments/1b51qrm/venting_about_my_shitty_life/', 'subreddit': 'mentalhealth', 'subreddit_id': 't5_2qirg', 'link_flair_text': 'Content Warning: Suicidal Thoughts / Self Harm', 'retrieved_on': 1709421194, 'post_id': 't3_1b51qrm'}, {'author': 'delvina_2', 'created_utc': 1709421881, 'permalink': '/r/mentalhealth/comm

In [12]:
collection.add(
    ids=ids,
    documents=documents,
    metadatas=metadatas,
)

## Batch upload to safeguard against network errors/api timeout

In [96]:
clean_df.to_csv("data/clean_df.csv", index=False)

In [15]:
BATCH_SIZE = 300
progress_file = "data/chroma_upload_progress.txt"


def load_progress(default_start: int = 0) -> int:
    """Return the next row index to start from.

    If the progress file does not exist, start at `default_start`.
    If it exists, it should contain a single integer: the next index.
    """
    if not os.path.exists(progress_file):
        return default_start

    with open(progress_file, "r") as f:
        value = f.read().strip()
        return int(value)



def save_progress(next_index: int) -> None:
    with open(progress_file, "w") as f:
        f.write(str(next_index))


In [18]:
start = load_progress()
n = len(clean_df)

print(f"Starting upload from index {start} out of {n} rows, batch size={BATCH_SIZE}.")

for batch_start in range(start, n, BATCH_SIZE):
    batch_end = min(batch_start + BATCH_SIZE, n)
    print(f"Preparing batch {batch_start}:{batch_end} (size={batch_end - batch_start})")

    batch_df = clean_df.iloc[batch_start:batch_end]

    ids = batch_df["post_id"].astype(str).tolist()
    documents = (batch_df["title"] + " | " + batch_df["selftext"]).tolist()
    metadatas = batch_df[
        [
            "author",
            "created_utc",
            "permalink",
            "subreddit",
            "subreddit_id",
            "link_flair_text",
            "retrieved_on",
            "post_id",
        ]
    ].to_dict(orient="records")

    try:
        print(f"Uploading batch starting at index {batch_start} with {len(ids)} documents...")
        collection.add(ids=ids, documents=documents, metadatas=metadatas)
        print(f"Successfully uploaded batch {batch_start}:{batch_end}.")
    except Exception as e:
        print(f"Error uploading batch {batch_start}:{batch_end}: {e}")
        break

    save_progress(batch_end)
    print(f"Progress saved. Next start index will be {batch_end}.")


Starting upload from index 143282 out of 143282 rows, batch size=300.
